# The Combine works, but read it right (v2, live recompute)
NFL Big Data Bowl 2027 · Open Track. Every number in the v2 writeup is recomputed here from the competition files (seed 2027).
If the competition data are not attached, the gate cell stops with an error: it never falls back to a saved figure.
Source: NFL Big Data Bowl 2027 competition data (CC BY-NC 4.0). Draft slot, pick and round are never used as variables.

## Shared code: rank partial correlation, Freedman–Lane permutation, bootstrap, BH, MDE, best-of-N

In [ ]:
# v2 analysis library (branch v2-review, 11/2026). Recomputes every v2 statistic LIVE from the nine competition CSVs.
# No precomputed result tables are read. Aggregate output only: no player names or IDs are printed or saved.
# Draft slot / pick / round is never a variable here. Seed 2027.
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats

SEED = 2027
MPH = 2.0455                      # yards/second -> mph
DEEP = ["GO", "POST", "CORNER", "WHEEL"]
BREAK = ["OUT", "IN", "HITCH", "SLANT"]
TIMES = {"forty", "ten_yd_split", "three_cone", "short_shuttle"}     # lower = better: sign flipped so + = better athlete

def find_comp(cands=("data", "../data", "/workspace/bdb2027/data")):
    for p in map(Path, cands):
        if (p / "combine_tracking.csv").exists():
            return p
    hits = list(Path("/kaggle/input").rglob("combine_tracking.csv")) if Path("/kaggle/input").exists() else []
    if hits:
        return hits[0].parent
    raise RuntimeError("Competition data are not attached (combine_tracking.csv not found). "
                       "GATE: FAIL. Attach the nfl-big-data-bowl-2027 competition data and rerun.")

# ------------------------------------------------------------------ statistics
def _design(df, cols):
    """Intercept + class dummies + ranks of any numeric controls; categorical controls as dummies."""
    parts = [np.ones((len(df), 1))]
    for c in cols:
        v = df[c]
        if c in ("draft_year", "role"):
            parts.append(pd.get_dummies(v.astype(str), drop_first=True).to_numpy(float))
        else:
            parts.append(stats.rankdata(v)[:, None])
    return np.hstack(parts)

def _res(v, Z):
    r = stats.rankdata(v)
    return r - Z @ np.linalg.lstsq(Z, r, rcond=None)[0]

def rank_partial(df, x, y, ctrl=("draft_year", "combine_weight", "combine_height")):
    d = df[[x, y, *ctrl]].dropna()
    Z = _design(d, ctrl)
    rx, ry = _res(d[x], Z), _res(d[y], Z)
    return float(np.corrcoef(rx, ry)[0, 1]), len(d), rx, ry, d

def perm_p(df, x, y, ctrl=("draft_year", "combine_weight", "combine_height"), P=10000, seed=SEED):
    """Freedman-Lane: shuffle the outcome's residuals (reduced model = controls only)."""
    r, n, rx, ry, d = rank_partial(df, x, y, ctrl)
    rng = np.random.default_rng(seed)
    rxs = (rx - rx.mean()) / rx.std(); ryc = ry - ry.mean()
    sims = np.array([abs(np.dot(rxs, rng.permutation(ryc))) for _ in range(P)]) / (len(rx) * ryc.std())
    return float((1 + (sims >= abs(r) - 1e-12).sum()) / (P + 1))

def boot_ci(df, x, y, ctrl=("draft_year", "combine_weight", "combine_height"), B=2000, seed=SEED, stat=None):
    d = df.reset_index(drop=True) if stat else df[[x, y, *ctrl]].dropna().reset_index(drop=True)
    rng = np.random.default_rng(seed); out = []
    f = stat or (lambda dd: rank_partial(dd, x, y, ctrl)[0])
    for _ in range(B):
        dd = d.iloc[rng.integers(0, len(d), len(d))]
        try:
            out.append(f(dd))
        except Exception:
            pass
    out = np.array(out); out = out[np.isfinite(out)]
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = p[o] * n / np.arange(1, n + 1); q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1); return out

def mde(n, k=3, alpha=0.05, power=0.8):
    """Smallest |rho| seen `power` of the time (Fisher z, 1.06 Spearman factor, k controls)."""
    z = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    return float(np.tanh(z * np.sqrt(1.06 / max(n - k - 3, 1))))

def zscore(r, n, k):
    return np.arctanh(np.clip(np.abs(r), 0, 0.999999)) * np.sqrt(max(n - k - 3, 1) / 1.06)

# ------------------------------------------------------------------ data
def load(COMP):
    pl = pd.read_csv(COMP / "players.csv", usecols=["nfl_id", "nfl_position", "draft_year", "draft_round"])
    pl["undrafted"] = pl.draft_round.isna()           # descriptive only; never a model variable
    pl = pl.drop(columns="draft_round")
    cr = pd.read_csv(COMP / "combine_results.csv").drop_duplicates("nfl_id").drop(columns="draft_year")
    for t in TIMES:
        cr[t] = -cr[t]
    games = pd.read_csv(COMP / "games.csv", usecols=["game_id", "season", "season_type"])
    reg = games[games.season_type == "REG"]
    pp = pd.read_csv(COMP / "player_play.csv", usecols=[
        "game_id", "play_id", "nfl_id", "lined_up_position", "pass_result", "play_nullified_by_penalty", "target",
        "rec_yards", "route_ran", "separation_at_pass_forward", "pressure_allowed", "time_to_pressure", "sack",
        "tackle_for_loss", "tackle", "assist"])
    pp = pp[(pp.play_nullified_by_penalty == "N") & pp.game_id.isin(reg.game_id)].merge(reg[["game_id", "season"]], on="game_id")
    ct = pd.read_csv(COMP / "combine_tracking.csv", usecols=["nfl_id", "entity_type", "time", "drill_type", "drill_name", "attempt", "s"])
    ct = ct[ct.entity_type == "PLAYER"]
    return pl, cr, pp, ct

def combine_tracking_features(ct):
    f = pd.DataFrame(index=pd.Index(ct.nfl_id.unique(), name="nfl_id"))
    f["c_top_mph"] = ct[ct.drill_type == "FORTY_YARD_DASH"].groupby("nfl_id").s.max() * MPH
    ol = ct[ct.drill_type == "SKILL_DRILLS_OL"].copy()
    ol["drill"] = ol.drill_name.str.replace("PASS_PRO-MIRROR_DRILL", "PASS_PRO_MIRROR_DRILL")
    ol = ol.sort_values(["nfl_id", "drill", "attempt", "time"])
    ol["ds"] = ol.groupby(["nfl_id", "drill", "attempt"]).s.diff()
    att = ol.groupby(["nfl_id", "drill", "attempt"]).agg(peak_s=("s", "max"), acc=("ds", "max"), dec=("ds", "min"))
    att["acc"] *= 10; att["dec"] *= -10        # largest 0.1-s speed gain / loss, yd/s^2 (10 Hz)
    per = att.groupby(["nfl_id", "drill"]).mean().unstack("drill")   # averaged over attempts
    short = {"FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE": "slide", "OL_PULL_DRILL_DEEP_SHORT_PULL_LEFT": "pull_short",
             "OL_PULL_DRILL_FOLD_BLOCK_RIGHT": "fold", "OL_PULL_DRILL_LONG_PULL_RIGHT": "pull_long",
             "PASS_PRO_MIRROR_DRILL": "mirror", "PASS_RUSH_DROP": "rush_drop", "SCREEN_DRILL": "screen"}
    per.columns = [f"ol_{short[d]}_{m}" for m, d in per.columns]
    return f.join(per)

def wr_play_speeds(COMP, ids, chunksize=2_000_000):
    """Per-play max speed (yd/s) from game tracking, for the given players."""
    parts = []
    for yr in (2023, 2024, 2025):
        for ch in pd.read_csv(COMP / f"game_tracking_{yr}.csv", usecols=["game_id", "play_id", "nfl_id", "s"],
                              dtype={"s": "float32"}, chunksize=chunksize):
            ch = ch[ch.nfl_id.isin(ids)]
            if len(ch):
                parts.append(ch.groupby(["game_id", "play_id", "nfl_id"], sort=False).s.max().reset_index())
    return pd.concat(parts).groupby(["game_id", "play_id", "nfl_id"], as_index=False).s.max()

# ------------------------------------------------------------------ receivers
def wr_table(pl, cr, pp, feat, plays):
    wr = pl[pl.nfl_position == "WR"].set_index("nfl_id")
    deep = pp[pp.nfl_id.isin(wr.index) & pp.route_ran.isin(DEEP)]
    dp = plays.merge(deep[["game_id", "play_id", "nfl_id", "season"]], on=["game_id", "play_id", "nfl_id"])
    dp = dp.sort_values(["nfl_id", "game_id", "play_id"])
    cnt = dp.groupby("nfl_id").size()
    T = wr.join(cr.set_index("nfl_id")).join(feat)
    T["deep_routes"] = cnt
    T["sunday_speed_mph"] = (dp.groupby("nfl_id").s.quantile(0.9) * MPH).where(cnt >= 20)
    dp = dp.join(T.draft_year, on="nfl_id")
    rk = dp[dp.season == dp.draft_year]; rc = rk.groupby("nfl_id").size()
    T["sunday_rookie"] = (rk.groupby("nfl_id").s.quantile(0.9) * MPH).where(rc >= 20)
    for k in (20, 30):
        h = dp.groupby("nfl_id").head(k); hc = h.groupby("nfl_id").size()
        T[f"sunday_first{k}"] = (h.groupby("nfl_id").s.quantile(0.9) * MPH).where(hc >= k)
    routes = pp[pp.nfl_id.isin(wr.index) & pp.route_ran.notna()].copy()
    routes["rec_yards"] = routes.rec_yards.fillna(0)
    g = routes.groupby("nfl_id")
    T["yards_per_route"] = g.rec_yards.sum() / g.size()
    dr = routes[routes.route_ran.isin(DEEP)].groupby("nfl_id")
    T["deep_yds_per_deep_route"] = dr.rec_yards.sum() / dr.size()
    T["deep_target_share"] = dr.target.mean()                      # targets per deep route
    T["sep_all_mean"] = g.separation_at_pass_forward.mean()
    T["break_y"] = routes[routes.route_ran.isin(BREAK)].groupby("nfl_id").separation_at_pass_forward.mean()
    return T

def funnel(T):
    a = len(T); b = T.c_top_mph.notna().sum(); c = (T.deep_routes >= 20).sum()
    S = T.dropna(subset=["c_top_mph", "sunday_speed_mph", "combine_weight", "combine_height"])
    return dict(listed=a, tracked_40=int(b), deep20=int(c), both=len(S),
                undrafted_in_sample=int(S.undrafted.sum()), undrafted_listed=int(T.undrafted.sum())), S

# ------------------------------------------------------------------ DL / OL / DB samples
def dl_table(pl, cr, pp, feat):
    dl = pl[pl.nfl_position.isin(["DE", "OLB", "DT", "NT"])].set_index("nfl_id")
    dl["role"] = np.where(dl.nfl_position.isin(["DE", "OLB"]), "Edge", "Interior")
    s = pp[pp.nfl_id.isin(dl.index) & pp.lined_up_position.isin(["EDGE", "INTERIOR_LINE"])]
    rush = s[s.pass_result.notna()]; g = rush.groupby("nfl_id")
    T = dl.join(cr.set_index("nfl_id")).join(feat)
    T["rush_snaps"] = g.size()
    T["pressure_rate"] = g.time_to_pressure.apply(lambda v: v.notna().mean())
    T["sack_rate"] = g.sack.apply(lambda v: v.fillna(0).sum()) / g.size()
    T["tfl_rate"] = s.groupby("nfl_id").tackle_for_loss.apply(lambda v: (v == 1).mean() if v.dtype != object else v.isin([True, "True", 1]).mean())
    return T[T.rush_snaps >= 20]

def ol_table(pl, cr, pp, feat):
    ol = pl[pl.nfl_position.isin(["T", "G", "C"])].set_index("nfl_id")
    ol["role"] = np.where(ol.nfl_position == "T", "Tackle", "Interior")
    s = pp[pp.nfl_id.isin(ol.index) & pp.pressure_allowed.notna()]
    g = s.groupby("nfl_id")
    T = ol.join(cr.set_index("nfl_id")).join(feat)
    T["pb_snaps"] = g.size()
    T["pressure_allowed_rate"] = g.pressure_allowed.apply(lambda v: v.astype(str).eq("True").mean())
    T["pressure_allowed_rate"] = -T["pressure_allowed_rate"]       # + = better (fewer pressures)
    return T[T.pb_snaps >= 100]

def db_table(pl, cr, pp, feat):
    db = pl[pl.nfl_position.isin(["CB", "SS", "FS", "DB"])].set_index("nfl_id")
    db["role"] = np.where(db.nfl_position.isin(["CB", "DB"]), "CB", "S")
    s = pp[pp.nfl_id.isin(db.index) & pp.lined_up_position.isin(["CB", "SLOT_CB", "HIGH_SAFETY", "BOX_SAFETY"])].copy()
    tb = lambda v: v.astype(str).isin(["True", "1", "1.0"])
    s["tk"] = tb(s.tackle) | tb(s.assist); s["tf"] = tb(s.tackle_for_loss)
    g = s.groupby("nfl_id")
    T = db.join(cr.set_index("nfl_id")).join(feat)
    T["def_snaps"] = g.size(); T["tackle_rate"] = g.tk.mean(); T["tfl_rate"] = g.tf.mean()
    return T[T.def_snaps >= 50]

# ------------------------------------------------------------------ families and best-of-N (Westfall-Young max-|z|)
def family(T, tests, outcomes, ctrl, min_n=25, P=10000):
    rows = []
    for t in tests:
        for o in outcomes:
            d = T[[t, o, *ctrl]].dropna()
            if len(d) < min_n:
                continue
            r, n, *_ = rank_partial(d, t, o, ctrl)
            rows.append(dict(test=t, outcome=o, n=n, rho=r, p=perm_p(d, t, o, ctrl, P=P)))
    F = pd.DataFrame(rows); F["q"] = bh(F.p); F["k"] = _design(T.dropna(subset=list(ctrl)), ctrl).shape[1] - 1
    F["z"] = [zscore(r, n, k) for r, n, k in zip(F.rho, F.n, F.k)]
    return F

def best_of_n(T, F, ctrl, S=2000, seed=SEED):
    """Shuffle the whole outcome block within draft class (players keep their tests and controls), recompute every
    cell; p_maxT = share of shuffles whose best |z| beats the cell's observed |z|. Also the k-th best luck bands."""
    rng = np.random.default_rng(seed)
    outs = sorted(F.outcome.unique()); tests = sorted(F.test.unique())
    base = T.dropna(subset=list(ctrl)).copy()
    idx_by_class = [np.where(base.draft_year.values == y)[0] for y in sorted(base.draft_year.unique())]
    O = base[outs].to_numpy()
    maxz, kth = [], []
    cells = list(zip(F.test, F.outcome))
    for _ in range(S):
        perm = np.arange(len(base))
        for ix in idx_by_class:
            perm[ix] = rng.permutation(ix)
        B = base.copy(); B[outs] = O[perm]
        zs = []
        for t, o in cells:
            d = B[[t, o, *ctrl]].dropna()
            r, n, *_ = rank_partial(d, t, o, ctrl)
            zs.append(zscore(r, n, F.k.iloc[0]))
        zs = np.sort(np.array(zs))[::-1]; maxz.append(zs[0]); kth.append(zs)
    maxz = np.array(maxz)
    F = F.copy(); F["p_best_of_n"] = [(1 + (maxz >= z - 1e-12).sum()) / (S + 1) for z in F.z]
    return F, np.array(kth)

# ------------------------------------------------------------------ Table 2: nested leave-one-class-out R^2 gain
def _nscore(v, cls):
    out = pd.Series(np.nan, index=v.index)
    for c in cls.unique():
        m = (cls == c) & v.notna(); r = stats.rankdata(v[m]); out[m] = stats.norm.ppf((r - 0.5) / m.sum())
    return out

def loco_gain(T, tests, y, size=("combine_weight", "combine_height"), extra=(), min_n=25):
    """Best Combine test chosen inside two training classes (by |rank partial|), scored on the third.
    Gain = R^2(size + test) - R^2(size only) on pooled held-out predictions; within-class normal scores."""
    d = T[[*tests, y, "draft_year", *size, *extra]].copy()
    d = d[d[y].notna()]
    for c in [*tests, y, *size]:
        d[c] = _nscore(d[c], d.draft_year).where(d[c].notna())
    preds0, preds1, ys = [], [], []
    for hold in sorted(d.draft_year.unique()):
        tr, te = d[d.draft_year != hold], d[d.draft_year == hold]
        best = max(tests, key=lambda t: abs(rank_partial(tr, t, y, ("draft_year", *size, *extra))[0]) if tr[t].notna().sum() >= min_n else -1)
        cols0 = [*size, *[e for e in extra]]
        trb, teb = tr.dropna(subset=[best, *size]), te.dropna(subset=[best, *size])
        def X(df, cols):
            parts = [np.ones(len(df))] + [df[c].to_numpy(float) if c != "role" else (df[c] == df[c].iloc[0]).to_numpy(float) for c in cols]
            return np.column_stack(parts)
        if "role" in extra:
            for df in (trb, teb):
                df["role"] = (df["role"] == "Tackle") | (df["role"] == "Edge") | (df["role"] == "CB")
            cols0 = [*size, "role"]
        def Xm(df, cols): return np.column_stack([np.ones(len(df))] + [df[c].to_numpy(float) for c in cols])
        b0 = np.linalg.lstsq(Xm(trb, cols0), trb[y].to_numpy(float), rcond=None)[0]
        b1 = np.linalg.lstsq(Xm(trb, cols0 + [best]), trb[y].to_numpy(float), rcond=None)[0]
        preds0.append(Xm(teb, cols0) @ b0); preds1.append(Xm(teb, cols0 + [best]) @ b1); ys.append(teb[y].to_numpy(float))
    p0, p1, yy = map(np.concatenate, (preds0, preds1, ys))
    sst = ((yy - yy.mean()) ** 2).sum()
    return float((((yy - p0) ** 2).sum() - ((yy - p1) ** 2).sum()) / sst), len(yy)

def loco_null(T, tests, y, S=300, seed=SEED, **kw):
    rng = np.random.default_rng(seed); obs, n = loco_gain(T, tests, y, **kw); sims = []
    for _ in range(S):
        P = T.copy()
        for c in P.draft_year.dropna().unique():
            m = (P.draft_year == c).to_numpy(); P.loc[m, y] = rng.permutation(P.loc[m, y].to_numpy())
        sims.append(loco_gain(P, tests, y, **kw)[0])
    sims = np.array(sims)
    return obs, n, float(np.percentile(sims, 95)), float((1 + (sims >= obs).sum()) / (S + 1))

In [ ]:
import json, time, warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"font.size": 9.5, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110, "savefig.dpi": 150})
CLS = {2023: "#009E73", 2024: "#E69F00", 2025: "#7570B3"}   # green/orange/purple = 2023/2024/2025 classes, always
BLUE, GREY, RED = "#0072B2", "#8a8a8a", "#D55E00"
ND = "NOT DETECTED (low power)"
SAVE = Path("/kaggle/working/figs") if Path("/kaggle/working").exists() else Path("figs"); SAVE.mkdir(parents=True, exist_ok=True)
R = {}                                  # every reported number lands here, then results_v2.json
SZ = ("draft_year", "combine_weight", "combine_height")
def cell(df, x, y, ctrl=SZ, P=10000, B=2000):
    r, n, *_ = rank_partial(df, x, y, ctrl); lo, hi = boot_ci(df, x, y, ctrl, B=B); p = perm_p(df, x, y, ctrl, P=P)
    return dict(rho=r, n=n, lo=lo, hi=hi, p=p, mde=mde(n, len(ctrl)))
import textwrap
def footer(fig, text):
    fig.text(.01, .01, "\n".join(textwrap.wrap(text, int(fig.get_size_inches()[0] * 17))), fontsize=7.5, color="#444", va="bottom")

## The gate: Combine tracking → regular-season game tracking (receivers)

In [ ]:
t0 = time.time()
COMP = find_comp()                       # raises RuntimeError -> GATE FAIL if the competition data are not attached
pl, cr, pp, ct = load(COMP)
feat = combine_tracking_features(ct)
plays = wr_play_speeds(COMP, set(pl.nfl_id[pl.nfl_position == "WR"]))
WRALL = wr_table(pl, cr, pp, feat, plays)
FUN, WR = funnel(WRALL); R["funnel"] = FUN
print("Funnel: {listed} receivers listed -> {tracked_40} with a tracked 40 -> {deep20} with 20+ deep routes -> {both} with both".format(**FUN))
print(f"Undrafted (descriptive only): {FUN['undrafted_in_sample']} of {FUN['both']} in the sample vs {FUN['undrafted_listed']} of {FUN['listed']} listed")
g = cell(WR, "c_top_mph", "sunday_speed_mph"); R["gate"] = g
WRF = family(WRALL, ["c_top_mph", "vertical", "broad_jump", "short_shuttle"], ["sunday_speed_mph", "sep_all_mean", "break_y"], SZ)
gq = float(WRF.query("test == 'c_top_mph' and outcome == 'sunday_speed_mph'").q.iloc[0]); R["gate"]["q"] = gq
print(f"Combine top speed vs Sunday Speed, size-adjusted: rho {g['rho']:+.2f} (95% CI {g['lo']:+.2f} to {g['hi']:+.2f}), n {g['n']}, permutation p {g['p']:.4f}, BH q {gq:.3f}")
ok = g["rho"] > 0 and g["lo"] > 0 and gq < 0.05
if not ok:
    raise RuntimeError("GATE: FAIL (live recompute does not support the Combine-to-game link)")
print("GATE (Combine tracking -> regular-season performance): PASS", f"[{time.time() - t0:.0f}s]")

## Table 1: who is in the sample, and does the receiver link survive other samples?

In [ ]:
T1 = {"Size-adjusted (class, weight, height)": g,
      "Class only (no size adjustment)": cell(WR, "c_top_mph", "sunday_speed_mph", ("draft_year",)),
      "Laser-timed 40 instead of tracked top speed": cell(WR, "forty", "sunday_speed_mph"),
      "Rookie-season routes only (20+)": cell(WRALL, "c_top_mph", "sunday_rookie"),
      "First 20 deep routes only": cell(WR, "c_top_mph", "sunday_first20"),
      "First 30 deep routes only": cell(WRALL, "c_top_mph", "sunday_first30")}
R["table1"] = T1
for k, v in T1.items():
    print(f"{k:45s} n {v['n']:3d}  rho {v['rho']:+.2f} ({v['lo']:+.2f}, {v['hi']:+.2f})")
x, y = WR.c_top_mph, WR.sunday_speed_mph
rng = np.random.default_rng(SEED); bs = [np.polyfit(x.values[i], y.values[i], 1)[0] for i in (rng.integers(0, len(WR), len(WR)) for _ in range(2000))]
R["slope"] = dict(b=float(np.polyfit(x, y, 1)[0]), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)), theil_sen=float(stats.theilslopes(y, x)[0]))
R["by_class"] = {}
for c in (2023, 2024, 2025):
    d = WR[WR.draft_year == c]; f = lambda dd: stats.spearmanr(dd.c_top_mph, dd.sunday_speed_mph)[0]
    lo, hi = boot_ci(d, "c_top_mph", "sunday_speed_mph", (), stat=f); R["by_class"][c] = dict(rho=float(f(d)), n=len(d), lo=lo, hi=hi)
print("slope", {k: round(v, 2) for k, v in R["slope"].items()}); print("by class", {k: (round(v["rho"], 2), v["n"]) for k, v in R["by_class"].items()})

In [ ]:
# Figure 1: receiver speed carries to Sunday (title has no "drafted": 7 of the 62 are undrafted)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
xs = np.linspace(x.min(), x.max(), 50); bb = []
rng = np.random.default_rng(SEED)
for _ in range(1000):
    i = rng.integers(0, len(WR), len(WR)); bb.append(np.polyval(np.polyfit(x.values[i], y.values[i], 1), xs))
a.fill_between(xs, *np.percentile(bb, [2.5, 97.5], axis=0), color="#ddd", lw=0)
a.plot(xs, np.polyval(np.polyfit(x, y, 1), xs), color="k", lw=1.8)
for c, col in CLS.items():
    m = WR.draft_year == c; a.scatter(x[m], y[m], s=24, color=col, edgecolor="white", lw=.4, label=f"{c} class (n={m.sum()})")
a.set_xlabel("Combine top speed in the 40-yard dash (mph, tracked at 10 Hz)"); a.set_ylabel("Sunday Speed (mph)\n90th-percentile top speed on deep routes")
a.set_title("A receiver's 40 speed shows up on Sunday", loc="left", fontweight="bold")
a.text(.02, .97, f"size-adjusted ρ {g['rho']:+.2f} [{g['lo']:+.2f}, {g['hi']:+.2f}], n {g['n']}\n{R['slope']['b']:+.2f} mph per mph [{R['slope']['lo']:.2f}, {R['slope']['hi']:.2f}]",
       transform=a.transAxes, va="top", fontsize=8.5)
a.legend(loc="lower right", fontsize=8, frameon=False)
rows = [(f"{c} class", v["rho"], v["lo"], v["hi"], v["n"], CLS[c]) for c, v in R["by_class"].items()]
rows.append(("All, class held fixed", T1["Class only (no size adjustment)"]["rho"], T1["Class only (no size adjustment)"]["lo"], T1["Class only (no size adjustment)"]["hi"], g["n"], "k"))
for i, (lab, r, lo, hi, n, col) in enumerate(rows[::-1]):
    b.plot([lo, hi], [i, i], color=col, lw=2); b.scatter([r], [i], color=col, s=40, zorder=3, marker="D" if col == "k" else "o")
    b.text(1.02, i, f"n={n}", va="center", fontsize=8)
b.set_yticks(range(len(rows))); b.set_yticklabels([r[0] for r in rows[::-1]]); b.axvline(0, color="k", ls=":", lw=.8); b.set_xlim(-.1, 1.1)
b.set_xlabel("ρ, 40 top speed vs Sunday Speed (95% CI)"); b.set_title("Same sign in every draft class", loc="left", fontweight="bold")
footer(fig, f"Sample: {g['n']} receivers (2023–25 classes; {FUN['undrafted_in_sample']} undrafted) with a tracked 40 and 20+ regular-season deep routes (go, post, corner, wheel). "
       "Line and band: straight-line fit, 1,000 player bootstraps. Right: within-class ρ, no size adjustment; bottom row holds class fixed.")
fig.tight_layout(rect=(0, .07, 1, 1)); fig.savefig(SAVE / "fig1_wr_speed_carries_over.png"); plt.show()

## Receivers: the NGS athleticism score, weight, and production

In [ ]:
N = {"NGS score, class only": cell(WR, "ngs_athleticism_score", "sunday_speed_mph", ("draft_year",)),
     "NGS score, size-adjusted": cell(WR, "ngs_athleticism_score", "sunday_speed_mph"),
     "Laser 40, size-adjusted": T1["Laser-timed 40 instead of tracked top speed"],
     "Tracked top speed, size-adjusted": g,
     "NGS score, top speed held fixed": cell(WR, "ngs_athleticism_score", "sunday_speed_mph", SZ + ("c_top_mph",)),
     "Top speed, NGS score held fixed": cell(WR, "c_top_mph", "sunday_speed_mph", SZ + ("ngs_athleticism_score",))}
R["ngs"] = N; R["ngs_weight"] = rank_partial(WR, "ngs_athleticism_score", "combine_weight", ("draft_year",))[0]
def dif(a, b2): return lambda dd: rank_partial(dd, a, "sunday_speed_mph", SZ)[0] - rank_partial(dd, b2, "sunday_speed_mph", SZ)[0]
W2 = WR.dropna(subset=["ngs_athleticism_score", "forty"])
R["diff_top_ngs"] = dict(d=dif("c_top_mph", "ngs_athleticism_score")(W2), ci=boot_ci(W2, "c_top_mph", "sunday_speed_mph", SZ, stat=dif("c_top_mph", "ngs_athleticism_score")))
R["diff_top_laser"] = dict(d=dif("c_top_mph", "forty")(W2), ci=boot_ci(W2, "c_top_mph", "sunday_speed_mph", SZ, stat=dif("c_top_mph", "forty")))
for k, v in N.items(): print(f"{k:35s} {v['rho']:+.2f} ({v['lo']:+.2f}, {v['hi']:+.2f}) p {v['p']:.3f}")
print("NGS vs weight (class fixed)", round(R["ngs_weight"], 2), "| top - NGS", R["diff_top_ngs"], "| top - laser", R["diff_top_laser"])
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.9), gridspec_kw={"width_ratios": [1.4, 1]})
for i, (k, v) in enumerate(list(N.items())[::-1]):
    col = BLUE if (v["lo"] > 0 and "NGS score" not in k.split(",")[0]) else GREY
    a.plot([v["lo"], v["hi"]], [i, i], color=col, lw=2); a.scatter([v["rho"]], [i], color=col, s=36, zorder=3); a.text(1.0, i, f"{v['rho']:+.2f}", va="center", fontsize=8)
a.set_yticks(range(len(N))); a.set_yticklabels(list(N)[::-1]); a.axvline(0, color="k", ls=":", lw=.8); a.set_xlim(-.45, 1.05)
a.set_xlabel(f"ρ with Sunday Speed (95% CI), n {g['n']}"); a.set_title("The NGS score adds nothing once top speed is known", loc="left", fontweight="bold")
for i, (lab, v) in enumerate([("Top speed − NGS score", R["diff_top_ngs"]), ("Top speed − laser 40", R["diff_top_laser"])][::-1]):
    col = BLUE if v["ci"][0] > 0 else GREY
    b.plot(v["ci"], [i, i], color=col, lw=2); b.scatter([v["d"]], [i], color=col, s=36, zorder=3); b.text(.62, i, f"{v['d']:+.2f}", va="center", fontsize=8)
b.set_yticks([0, 1]); b.set_yticklabels(["Top speed − laser 40", "Top speed − NGS score"]); b.axvline(0, color="k", ls=":", lw=.8); b.set_xlim(-.2, .7); b.set_ylim(-.6, 1.6)
b.set_xlabel("Difference in size-adjusted ρ\n(paired bootstrap)"); b.set_title("Top speed beats the score", loc="left", fontweight="bold")
footer(fig, "Blue: interval above zero. Grey: interval includes zero, or the NGS score. Same receivers throughout; 'held fixed' adds that measure as a control. NGS athleticism score: column in combine_results.csv.")
fig.tight_layout(rect=(0, .07, 1, 1)); fig.savefig(SAVE / "fig2_ngs_vs_tracked_speed.png"); plt.show()

In [ ]:
# Weight: heavier receivers run slower than their own 40 says (Figure 3, regenerated from its spec)
R["wt_vs_top"] = cell(WR, "combine_weight", "c_top_mph", ("draft_year", "combine_height"))
R["wt_partial"] = cell(WR, "combine_weight", "sunday_speed_mph", ("draft_year", "combine_height", "c_top_mph"))
def wslope(dd):
    X = np.column_stack([np.ones(len(dd)), dd.c_top_mph, dd.combine_weight, dd.combine_height, pd.get_dummies(dd.draft_year.astype(str), drop_first=True).to_numpy(float)])
    return float(np.linalg.lstsq(X, dd.sunday_speed_mph.to_numpy(float), rcond=None)[0][2] * 10)
R["wt_slope10"] = dict(b=wslope(WR), ci=boot_ci(WR, "combine_weight", "sunday_speed_mph", (), stat=wslope))
R["wt_slope_by_class"] = {}
for c in (2023, 2024, 2025):
    d = WR[WR.draft_year == c]; X = np.column_stack([np.ones(len(d)), d.c_top_mph, d.combine_weight])
    R["wt_slope_by_class"][c] = float(np.linalg.lstsq(X, d.sunday_speed_mph.to_numpy(float), rcond=None)[0][2] * 10)
print("weight vs top speed", {k: round(v, 2) for k, v in R["wt_vs_top"].items()}); print("weight | top speed", {k: round(v, 3) for k, v in R["wt_partial"].items()})
print("mph per 10 lb", R["wt_slope10"], "by class", R["wt_slope_by_class"])
# leftover = Sunday Speed minus what top speed, height and class predict (mph)
X = np.column_stack([np.ones(len(WR)), WR.c_top_mph, WR.combine_height, pd.get_dummies(WR.draft_year.astype(str), drop_first=True).to_numpy(float)])
left = WR.sunday_speed_mph - X @ np.linalg.lstsq(X, WR.sunday_speed_mph.to_numpy(float), rcond=None)[0]
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.5, 1]})
for c, col in CLS.items():
    m = WR.draft_year == c; a.scatter(WR.combine_weight[m], left[m], s=24, color=col, edgecolor="white", lw=.4, label=f"{c} class")
xx = np.linspace(WR.combine_weight.min(), WR.combine_weight.max(), 2); bw = np.polyfit(WR.combine_weight, left, 1)
a.plot(xx, np.polyval(bw, xx), color=BLUE, lw=2); a.axhline(0, color="k", lw=.6)
a.set_xlabel("Combine weight (lb)"); a.set_ylabel("Sunday Speed vs what his 40 top speed predicts (mph)")
a.set_title("Heavier receivers run slower on Sunday than their 40 says", loc="left", fontweight="bold")
s = R["wt_slope10"]; a.text(.02, .04, f"{s['b']:+.2f} mph per 10 lb [{s['ci'][0]:+.2f}, {s['ci'][1]:+.2f}]\npartial ρ {R['wt_partial']['rho']:+.2f}, n {R['wt_partial']['n']}", transform=a.transAxes, fontsize=8.5)
a.legend(fontsize=8, frameon=False, loc="upper right")
cls = list(R["wt_slope_by_class"].items())
b.barh([f"{c} class" for c, _ in cls][::-1], [v for _, v in cls][::-1], color=[CLS[c] for c, _ in cls][::-1])
b.axvline(0, color="k", lw=.8); b.set_xlabel("mph per 10 lb, at the same 40 top speed"); b.set_title("Same sign in every class", loc="left", fontweight="bold")
footer(fig, "Holding 40 top speed, height and class fixed. An association: the data cannot say whether weight, role, or how a receiver is used on deep routes explains it.")
fig.tight_layout(rect=(0, .06, 1, 1)); fig.savefig(SAVE / "fig3_heavy_receivers_slower.png"); plt.show()

In [ ]:
# Production: does Sunday Speed buy production? (class held fixed; top speed size-adjusted)
R["prod"] = {o: cell(WRALL, "sunday_speed_mph", o, ("draft_year",)) for o in ["yards_per_route", "deep_yds_per_deep_route", "deep_target_share"]}
R["prod_top"] = {o: cell(WRALL, "c_top_mph", o) for o in ["yards_per_route", "deep_yds_per_deep_route", "deep_target_share"]}
for k in R["prod"]: print(k, {kk: round(vv, 2) for kk, vv in R["prod"][k].items()}, "| top speed", round(R["prod_top"][k]["rho"], 2))

## Defensive line: how much of the drill link is role and size? (as-reported rates)

In [ ]:
DL = dl_table(pl, cr, pp, feat)
DLT = ["short_shuttle", "three_cone", "forty", "ten_yd_split", "c_top_mph", "broad_jump", "vertical"]; DLO = ["pressure_rate", "sack_rate", "tfl_rate"]
M = [("draft_year",), ("draft_year", "role"), ("draft_year", "role", "combine_weight", "combine_height")]
def ladder(D):
    rows = []
    for t in DLT:
        for o in DLO:
            d = D[[t, o, *M[2]]].dropna()
            if len(d) >= 25: rows.append([rank_partial(d, t, o, m)[0] for m in M] + [len(d)])
    return np.array(rows)
L = ladder(DL); med = np.median(L[:, :3], axis=0)
rng = np.random.default_rng(SEED); sh = []
for _ in range(1000):
    Lb = ladder(DL.iloc[rng.integers(0, len(DL), len(DL))]); m = np.median(Lb[:, :3], axis=0); sh.append(1 - m[2] / m[0])
R["dl"] = dict(n_players=len(DL), cells=len(L), n_lo=int(L[:, 3].min()), n_hi=int(L[:, 3].max()), med=med.tolist(), share=float(1 - med[2] / med[0]),
               share_role=float(1 - med[1] / med[0]), share_ci=np.percentile(sh, [2.5, 97.5]).tolist(), undrafted=int(DL.undrafted.sum()))
SZR = ("draft_year", "role", "combine_weight", "combine_height")
R["dl40p"] = cell(DL, "forty", "pressure_rate", SZR)
R["dl_role"] = {r: cell(DL[DL.role == r], "forty", "pressure_rate") for r in ("Edge", "Interior")}
R["dl_wt_pressure"] = cell(DL, "combine_weight", "pressure_rate", ("draft_year", "role", "combine_height"))
print({k: v for k, v in R["dl"].items()}); print("40 -> pressure", R["dl40p"]); print("within role", R["dl_role"]); print("weight -> pressure", R["dl_wt_pressure"])
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [1.2, 1]})
lab = {"short_shuttle": "Short shuttle", "three_cone": "3-cone", "forty": "40-yd dash", "ten_yd_split": "10-yd split", "c_top_mph": "40 top speed", "broad_jump": "Broad jump", "vertical": "Vertical"}
for i, t in enumerate(DLT[::-1]):
    d = DL[[t, "pressure_rate", *M[2]]].dropna(); v = [rank_partial(d, t, "pressure_rate", m)[0] for m in M]
    a.plot([min(v), max(v)], [i, i], color="#ccc", lw=2, zorder=1)
    for vv, col in zip(v, [GREY, "#E69F00", BLUE]): a.scatter([vv], [i], color=col, s=40, zorder=3)
    lab[t] = lab[t] + f" (n={len(d)})"
a.set_yticks(range(len(DLT))); a.set_yticklabels([lab[t] for t in DLT[::-1]]); a.axvline(0, color="k", ls=":", lw=.8); a.set_xlim(-.45, .85)
a.set_xlabel("Rank partial ρ with pressure rate (as reported)\n+ = better athlete, more pressures")
a.set_title(f"Most of the line link is edge vs interior\nmedian of {len(L)} cells: {med[0]:+.2f} → {med[1]:+.2f} (+role) → {med[2]:+.2f} (+size)", loc="left", fontweight="bold", fontsize=9.5)
a.scatter([], [], color=GREY, label="class only"); a.scatter([], [], color="#E69F00", label="+ role"); a.scatter([], [], color=BLUE, label="+ weight, height"); a.legend(fontsize=8, frameon=False, loc="lower left")
pairs = [("forty", "pressure_rate", "40-yd → pressure"), ("ten_yd_split", "pressure_rate", "10-yd → pressure"), ("forty", "sack_rate", "40-yd → sack")]
yl = []
for j, (t, o, lb) in enumerate(pairs[::-1]):
    for k, (r, col) in enumerate([("Edge", "#8c564b"), ("Interior", "#e377c2")]):
        v = cell(DL[DL.role == r], t, o, P=5000, B=1000); yy = j * 2.6 + k
        b.plot([v["lo"], v["hi"]], [yy, yy], color=col, lw=2); b.scatter([v["rho"]], [yy], color=col, s=34, zorder=3)
        b.text(1.02, yy, f"{r} n={v['n']} p" + ("<0.01" if v['p'] < .01 else f"={v['p']:.2f}"), va="center", fontsize=7.5); yl.append((yy + .5 if k == 0 else None, lb))
b.set_yticks([j * 2.6 + .5 for j in range(len(pairs))]); b.set_yticklabels([p[2] for p in pairs[::-1]]); b.axvline(0, color="k", ls=":", lw=.8); b.set_xlim(-.6, 1.0)
b.set_xlabel("Within-role rank partial ρ (95% CI)\nclass, weight, height held fixed"); b.set_title("Inside each role the link is weaker or noisy", loc="left", fontweight="bold", fontsize=9.5)
footer(fig, f"{len(DL)} defensive linemen (2023–25 classes; {R['dl']['undrafted']} undrafted) with 20+ pass-rush snaps. Pressure rate = pressures per pass-rush snap, as reported (not shrunk). Right: brown edge, pink interior; p = permutation p within role, unadjusted.")
fig.tight_layout(rect=(0, .07, .93, 1)); fig.savefig(SAVE / "fig4_dl_ladder.png"); plt.show()

## Offensive line: slide-drill acceleration and pressures allowed

In [ ]:
OL = ol_table(pl, cr, pp, feat); X0, Y0 = "ol_slide_acc", "pressure_allowed_rate"   # Y0 is -(rate): + = fewer pressures
o = cell(OL, X0, Y0, SZR); R["ol"] = dict(o, rho_rate=-o["rho"], lo_rate=-o["hi"], hi_rate=-o["lo"], undrafted=int(OL.undrafted.sum()),
                                          snaps_med={int(c): float(v) for c, v in OL.groupby("draft_year").pb_snaps.median().items()})
R["ol_class"] = {}
for c in (2023, 2024, 2025):
    d = OL[OL.draft_year == c][[X0, Y0]].dropna(); f = lambda dd: stats.spearmanr(dd[X0], -dd[Y0])[0]
    lo, hi = boot_ci(d, X0, Y0, (), stat=f); R["ol_class"][c] = dict(rho=float(f(d)), n=len(d), lo=lo, hi=hi)
R["ol_2324"] = cell(OL[OL.draft_year < 2025], X0, Y0, SZR)
dd = OL[[X0, Y0, "draft_year"]].dropna()
def cdiff(df):
    a = df[df.draft_year == 2025]; return stats.spearmanr(a[X0], a[Y0])[0] - np.mean([stats.spearmanr(df[df.draft_year == k][X0], df[df.draft_year == k][Y0])[0] for k in (2023, 2024)])
rng = np.random.default_rng(SEED); obs = cdiff(dd); sims = []
for _ in range(10000):
    e = dd.copy(); e["draft_year"] = rng.permutation(e.draft_year.values); sims.append(cdiff(e))
R["ol_class_p"] = float((1 + np.sum(np.abs(sims) >= abs(obs))) / 10001)
print("OL", {k: v for k, v in R["ol"].items()}); print("by class", R["ol_class"]); print("2023+24", R["ol_2324"]["rho"], "class-difference p", R["ol_class_p"])
fig, axs = plt.subplots(1, 4, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 1, 1, 1.35]})
for a, c in zip(axs[:3], (2023, 2024, 2025)):
    d = OL[OL.draft_year == c]; a.scatter(d[X0], -100 * d[Y0], s=22, color=CLS[c], edgecolor="white", lw=.4)
    bfit = np.polyfit(d[X0], -100 * d[Y0], 1); xx = np.linspace(d[X0].min(), d[X0].max(), 2); a.plot(xx, np.polyval(bfit, xx), color=RED if c == 2025 else "k", lw=1.6)
    v = R["ol_class"][c]; a.set_title(f"{c} (n={v['n']}): ρ {v['rho']:+.2f}\n[{v['lo']:+.2f}, {v['hi']:+.2f}]", loc="left", fontsize=9)
    a.set_xlabel("Slide-drill peak acceleration (yd/s²)")
axs[0].set_ylabel("Pressures allowed\nper pass-block snap (%)")
a = axs[3]; rows = [("All 3 classes + size, role", R["ol"]["rho_rate"], R["ol"]["lo_rate"], R["ol"]["hi_rate"], R["ol"]["n"], BLUE),
                    ("2023+2024 + size, role", -R["ol_2324"]["rho"], -R["ol_2324"]["hi"], -R["ol_2324"]["lo"], R["ol_2324"]["n"], "k"),
                    ("2025 only, raw", R["ol_class"][2025]["rho"], R["ol_class"][2025]["lo"], R["ol_class"][2025]["hi"], R["ol_class"][2025]["n"], RED)]
for i, (lb, r, lo, hi, n, col) in enumerate(rows[::-1]):
    a.plot([lo, hi], [i, i], color=col, lw=2); a.scatter([r], [i], color=col, s=36, zorder=3); a.text(.62, i, f"{r:+.2f} (n={n})", va="center", fontsize=8)
a.set_yticks(range(3)); a.set_yticklabels([r[0] for r in rows[::-1]], fontsize=8); a.axvline(0, color="k", ls=":", lw=.8); a.set_xlim(-1, .6)
a.set_xlabel("ρ with pressure rate allowed\n(− = more acceleration, fewer pressures)"); a.set_title("Strong in two classes, absent in 2025", loc="left", fontweight="bold", fontsize=9)
fig.suptitle("Offensive line: slide-drill acceleration went with fewer pressures allowed in 2023–24, not in 2025", x=.01, ha="left", fontweight="bold")
footer(fig, f"{R['ol']['n']} offensive linemen (2023–25 classes; {R['ol']['undrafted']} undrafted) with 100+ regular-season pass-block snaps. Peak acceleration = largest 0.1-s speed gain, averaged over attempts. "
       f"2025 vs other classes: permutation p {R['ol_class_p']:.3f}. The 2023+2024 restriction was chosen after seeing the data. Median snaps: " + ", ".join(f"{k} {v:.0f}" for k, v in R['ol']['snaps_med'].items()) + ".")
fig.tight_layout(rect=(0, .08, 1, .93)); fig.savefig(SAVE / "fig5_ol_slide_reversal.png"); plt.show()

## How many tests, and what survives best-of-N luck (Westfall–Young max-|z|, 2,000 within-class shuffles)

In [ ]:
DB = db_table(pl, cr, pp, feat)
OLT = ["c_top_mph", "forty", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps"] + [c for c in OL.columns if c.startswith("ol_")]
FAMS = {"Receivers": (WRALL, ["c_top_mph", "vertical", "broad_jump", "short_shuttle"], ["sunday_speed_mph", "sep_all_mean", "break_y"], SZ),
        "Defensive line": (DL, ["c_top_mph", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle"], DLO, SZR),
        "Offensive line": (OL, OLT, [Y0], SZR),
        "Defensive backs": (DB, ["c_top_mph", "ten_yd_split", "vertical", "broad_jump", "three_cone", "short_shuttle"], ["tackle_rate", "tfl_rate"], SZ + ("role",))}
FR, KTH = {}, {}
for k, (T, te, ou, c) in FAMS.items():
    F = family(T, te, ou, c); F, kth = best_of_n(T, F, c, S=2000); FR[k], KTH[k] = F, kth
R["families"] = {k: dict(cells=len(F), p05=int((F.p < .05).sum()), expected=round(.05 * len(F), 1), q05=int((F.q < .05).sum()),
                         survive=int((F.p_best_of_n < .05).sum()), best=F.sort_values("p").iloc[0][["test", "outcome", "n", "rho", "p", "q", "p_best_of_n"]].to_dict()) for k, F in FR.items()}
R["families_total"] = dict(cells=sum(v["cells"] for v in R["families"].values()), p05=sum(v["p05"] for v in R["families"].values()),
                           expected=round(.05 * sum(v["cells"] for v in R["families"].values()), 1), survive=sum(v["survive"] for v in R["families"].values()))
R["db"] = dict(n_players=len(DB), mde=mde(int(np.median(FR["Defensive backs"].n)), 4), n_med=int(np.median(FR["Defensive backs"].n)))
for k, v in R["families"].items(): print(k, v)
print("TOTAL", R["families_total"])
fig, axs = plt.subplots(1, 4, figsize=(12.5, 4), sharey=True)
for a, (k, F) in zip(axs, FR.items()):
    z = np.sort(F.z.values)[::-1]; kth = KTH[k]; r = np.arange(1, len(z) + 1)
    a.fill_between(r, np.percentile(kth, 5, axis=0), np.percentile(kth, 95, axis=0), color="#ddd", lw=0); a.plot(r, np.median(kth, axis=0), color=GREY, lw=1)
    surv = np.sort(F.z[F.p_best_of_n < .05].values)[::-1]
    a.scatter(r, z, color=["#0072B2" if zz in surv else "k" for zz in z], s=[60 if zz in surv else 18 for zz in z], zorder=3)
    v = R["families"][k]; a.set_title(f"{k}: {v['cells']} cells\n{v['p05']} with p<.05 ({v['expected']} by chance)", loc="left", fontsize=9, fontweight="bold")
    a.set_xlabel("Rank of cell (strongest = 1)")
axs[0].set_ylabel("Evidence |z| (rank partial ρ → z)")
fig.suptitle("Only two links beat best-of-N luck: receiver speed and offensive-line slide-drill acceleration", x=.01, ha="left", fontweight="bold")
footer(fig, "Each dot is one Combine test × outcome cell (25+ players), size-adjusted. Grey band: luck, 5th–95th percentile of the k-th best cell when the whole outcome block is shuffled within draft class (2,000 shuffles). Blue: survives best-of-N (p<.05).")
fig.tight_layout(rect=(0, .07, 1, .93)); fig.savefig(SAVE / "fig6_best_of_n_null.png"); plt.show()

## Table 2: held-out effect sizes (best test chosen inside two classes, scored on the third; 300-shuffle luck range)

In [ ]:
WRT = ["c_top_mph", "forty", "vertical", "broad_jump", "short_shuttle", "ten_yd_split"]
T2 = {"Receiver Sunday Speed": loco_null(WRALL, WRT, "sunday_speed_mph"),
      "Receiver separation, break routes": loco_null(WRALL, WRT, "break_y"),
      "Receiver separation, all routes": loco_null(WRALL, WRT, "sep_all_mean"),
      "Defensive-line pressure rate": loco_null(DL, ["forty", "ten_yd_split", "c_top_mph", "vertical", "broad_jump"], "pressure_rate", extra=("role",)),
      "Offensive-line pressures allowed": loco_null(OL, OLT, Y0, extra=("role",))}
R["table2"] = {k: dict(gain=v[0], n=v[1], luck95=v[2], p=v[3]) for k, v in T2.items()}
for k, v in R["table2"].items(): print(f"{k:36s} n {v['n']:3d} gain {v['gain']:+.3f} luck95 {v['luck95']:+.3f} p {v['p']:.3f}")

## Tight ends, and the results file

In [ ]:
te = pl[pl.nfl_position == "TE"].set_index("nfl_id").join(cr.set_index("nfl_id")).join(feat)
te_routes = pp[pp.nfl_id.isin(te.index) & pp.route_ran.notna()].groupby("nfl_id").size()
R["te"] = dict(listed=len(te), tracked40_and_20routes=int(((te.c_top_mph.notna()) & (te_routes.reindex(te.index).fillna(0) >= 20)).sum()))
R["te"]["mde"] = mde(R["te"]["tracked40_and_20routes"], 3)
print("TE", R["te"])
def _j(o):
    if isinstance(o, dict): return {str(k): _j(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_j(v) for v in o]
    if isinstance(o, (np.floating, float)): return float(o)
    if isinstance(o, (np.integer, int)): return int(o)
    return o
OUTJ = (SAVE.parent / "results_v2.json"); OUTJ.write_text(json.dumps(_j(R), indent=1))
print("figures:", sorted(p.name for p in SAVE.glob("fig*.png")), "| results:", OUTJ, f"| total {time.time() - t0:.0f}s")
assert len(list(SAVE.glob("fig*.png"))) == 6